# DREGON-LibriMix Dataset Inspector

This notebook allows you to inspect samples from the generated DREGON-LibriMix dataset:
- View spectrograms of mixture, vocals, and noise
- View motor speeds (RPS) aligned in time
- Listen to audio samples

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

In [10]:
# Configuration
DATASET_PATH = "./datasets/DREGON-LM-test/train"  # Change to valid for validation set
SAMPLE_RATE = 16000  # Audio sample rate
RPS_RATE = 1000  # Approximate RPS sample rate (native motor rate)
N_FFT = 2048
HOP_LENGTH = 512

In [ ]:
def get_sample_list(dataset_path):
    """Get list of sample folders in the dataset."""
    dataset_path = Path(dataset_path)
    samples = sorted([d.name for d in dataset_path.iterdir() if d.is_dir() and d.name.startswith('sample_')])
    return samples

def load_sample(dataset_path, sample_name):
    """Load all data for a sample."""
    sample_path = Path(dataset_path) / sample_name

    data = {}

    # Load audio files
    for audio_type in ['mixture', 'vocals', 'noise']:
        audio_path = sample_path / f"{audio_type}.wav"
        if audio_path.exists():
            audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE)
            data[audio_type] = audio
            data['sample_rate'] = sr

    # Load RPS data
    rps_path = sample_path / "rps.npy"
    if rps_path.exists():
        data['rps'] = np.load(rps_path)  # Shape: (4, n_samples)

    return data

In [ ]:
def plot_sample(data, figsize=(14, 12)):
    """Plot spectrograms and RPS data aligned in time."""

    # Get audio duration
    audio = data.get('mixture', data.get('vocals', data.get('noise')))
    if audio is None:
        print("No audio data found!")
        return

    duration = len(audio) / SAMPLE_RATE

    # Create figure with subplots
    n_audio_plots = sum(1 for k in ['mixture', 'vocals', 'noise'] if k in data)
    has_rps = 'rps' in data
    n_plots = n_audio_plots + (1 if has_rps else 0)

    fig, axes = plt.subplots(n_plots, 1, figsize=figsize, sharex=True)
    if n_plots == 1:
        axes = [axes]

    plot_idx = 0

    # Plot spectrograms
    for audio_type, title in [('mixture', 'Mixture'), ('vocals', 'Vocals (Clean Speech)'), ('noise', 'Noise (Drone)')]:
        if audio_type not in data:
            continue

        audio = data[audio_type]

        # Compute spectrogram
        D = librosa.stft(audio, n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

        # Plot
        ax = axes[plot_idx]
        img = librosa.display.specshow(
            S_db,
            sr=SAMPLE_RATE,
            hop_length=HOP_LENGTH,
            x_axis='time',
            y_axis='hz',
            ax=ax,
            cmap='magma'
        )
        ax.set_title(f'{title} Spectrogram')
        ax.set_ylabel('Frequency (Hz)')
        fig.colorbar(img, ax=ax, format='%+2.0f dB')

        plot_idx += 1

    # Plot RPS data
    if has_rps:
        ax = axes[plot_idx]
        rps = data['rps']  # Shape: (4, n_samples)
        n_rotors, n_samples = rps.shape

        # Create time axis for RPS
        # RPS is at native motor rate, align to audio duration
        rps_time = np.linspace(0, duration, n_samples)

        # Plot each rotor
        colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
        for i in range(n_rotors):
            ax.plot(rps_time, rps[i], label=f'Rotor {i+1}', color=colors[i], alpha=0.8, linewidth=1)

        ax.set_title('Motor Speeds (RPS)')
        ax.set_ylabel('Rotations per Second')
        ax.set_xlabel('Time (s)')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(0, duration)

    plt.tight_layout()
    plt.show()

    # Print RPS statistics
    if has_rps:
        rps = data['rps']
        print(f"\nRPS Statistics:")
        print(f"  Shape: {rps.shape} (n_rotors, n_samples)")
        print(f"  Duration: {duration:.2f}s, RPS samples: {rps.shape[1]}, Effective rate: {rps.shape[1]/duration:.1f} Hz")
        for i in range(rps.shape[0]):
            print(f"  Rotor {i+1}: min={rps[i].min():.1f}, max={rps[i].max():.1f}, mean={rps[i].mean():.1f}, std={rps[i].std():.1f}")

In [ ]:
def play_audio(data):
    """Create audio players for all audio types."""
    print("Audio Players:")
    print("=" * 50)

    for audio_type, title in [('mixture', 'Mixture'), ('vocals', 'Vocals (Clean Speech)'), ('noise', 'Noise (Drone)')]:
        if audio_type in data:
            print(f"\n{title}:")
            display(ipd.Audio(data[audio_type], rate=SAMPLE_RATE))

In [11]:
# Get list of samples
import os

print(os.getcwd())
samples = get_sample_list(DATASET_PATH)
print(f"Found {len(samples)} samples in {DATASET_PATH}")
print(f"Samples: {samples[:10]}{'...' if len(samples) > 10 else ''}")

/home/flyingleafe/Research/PhD/projects/Edge-BS-RoFormer-DroneNoise-LibriMix
Found 50 samples in ./datasets/DREGON-LM-test/train
Samples: ['sample_00000', 'sample_00001', 'sample_00002', 'sample_00003', 'sample_00004', 'sample_00005', 'sample_00006', 'sample_00007', 'sample_00008', 'sample_00009']...


In [ ]:
# Interactive sample selector
def inspect_sample(sample_name, dataset_split='train'):
    """Load and display a sample."""
    dataset_path = f"./datasets/DREGON-LM-test/{dataset_split}"

    print(f"Loading sample: {sample_name} from {dataset_split}")
    print("=" * 50)

    data = load_sample(dataset_path, sample_name)

    # Print basic info
    if 'mixture' in data:
        duration = len(data['mixture']) / SAMPLE_RATE
        print(f"Audio duration: {duration:.2f}s")
        print(f"Sample rate: {SAMPLE_RATE} Hz")

    # Plot spectrograms and RPS
    plot_sample(data)

    # Audio players
    play_audio(data)

    return data

In [ ]:
# Create interactive widget
train_samples = get_sample_list("./datasets/DREGON-LM-test/train")
valid_samples = get_sample_list("./datasets/DREGON-LM-test/valid")

split_widget = widgets.Dropdown(
    options=['train', 'valid'],
    value='train',
    description='Split:'
)

sample_widget = widgets.Dropdown(
    options=train_samples,
    value=train_samples[0] if train_samples else None,
    description='Sample:'
)

def update_samples(*args):
    """Update sample list when split changes."""
    if split_widget.value == 'train':
        sample_widget.options = train_samples
    else:
        sample_widget.options = valid_samples

split_widget.observe(update_samples, 'value')

interact(inspect_sample, sample_name=sample_widget, dataset_split=split_widget);

interactive(children=(Dropdown(description='Sample:', options=('sample_00000', 'sample_00001', 'sample_00002',…

## Manual Sample Inspection

You can also manually inspect a specific sample:

In [ ]:
# Manual inspection - change sample name as needed
# data = inspect_sample('sample_00000', 'train')

Loading sample: sample_00000 from train
No audio data found!
Audio Players:


## Batch Statistics

View statistics across multiple samples:

In [ ]:
def compute_batch_stats(dataset_path, max_samples=None):
    """Compute statistics across all samples."""
    samples = get_sample_list(dataset_path)
    if max_samples:
        samples = samples[:max_samples]

    all_rps_means = []
    all_rps_stds = []
    all_durations = []
    all_snrs = []

    for sample_name in samples:
        data = load_sample(dataset_path, sample_name)

        if 'mixture' in data:
            duration = len(data['mixture']) / SAMPLE_RATE
            all_durations.append(duration)

        if 'rps' in data:
            rps = data['rps']
            all_rps_means.append(rps.mean(axis=1))  # Mean per rotor
            all_rps_stds.append(rps.std(axis=1))    # Std per rotor

        # Estimate SNR (speech to noise ratio)
        if 'vocals' in data and 'noise' in data:
            vocals_power = np.mean(data['vocals'] ** 2)
            noise_power = np.mean(data['noise'] ** 2)
            if noise_power > 0:
                snr = 10 * np.log10(vocals_power / noise_power)
                all_snrs.append(snr)

    print(f"Dataset Statistics ({len(samples)} samples)")
    print("=" * 50)

    if all_durations:
        print(f"\nAudio Duration:")
        print(f"  Mean: {np.mean(all_durations):.2f}s")
        print(f"  Std:  {np.std(all_durations):.2f}s")
        print(f"  Range: [{np.min(all_durations):.2f}s, {np.max(all_durations):.2f}s]")

    if all_rps_means:
        rps_means = np.array(all_rps_means)  # (n_samples, 4)
        rps_stds = np.array(all_rps_stds)
        print(f"\nRPS Statistics (per rotor):")
        for i in range(4):
            print(f"  Rotor {i+1}: mean={rps_means[:, i].mean():.1f} +/- {rps_means[:, i].std():.1f} RPS")

    if all_snrs:
        print(f"\nSNR (Speech-to-Noise Ratio):")
        print(f"  Mean: {np.mean(all_snrs):.1f} dB")
        print(f"  Std:  {np.std(all_snrs):.1f} dB")
        print(f"  Range: [{np.min(all_snrs):.1f} dB, {np.max(all_snrs):.1f} dB]")

    return {
        'durations': all_durations,
        'rps_means': all_rps_means,
        'rps_stds': all_rps_stds,
        'snrs': all_snrs
    }

In [ ]:
# Compute stats for training set
train_stats = compute_batch_stats("../datasets/DREGON-LM-test/train")

In [ ]:
# Plot histograms
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

if train_stats['durations']:
    axes[0].hist(train_stats['durations'], bins=20, edgecolor='black')
    axes[0].set_xlabel('Duration (s)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Audio Duration Distribution')

if train_stats['rps_means']:
    rps_means = np.array(train_stats['rps_means'])
    for i in range(4):
        axes[1].hist(rps_means[:, i], bins=20, alpha=0.5, label=f'Rotor {i+1}')
    axes[1].set_xlabel('Mean RPS')
    axes[1].set_ylabel('Count')
    axes[1].set_title('RPS Distribution (per sample)')
    axes[1].legend()

if train_stats['snrs']:
    axes[2].hist(train_stats['snrs'], bins=20, edgecolor='black')
    axes[2].set_xlabel('SNR (dB)')
    axes[2].set_ylabel('Count')
    axes[2].set_title('SNR Distribution')

plt.tight_layout()
plt.show()